In [1]:
import sys
sys.path.append('/home/nicolo_b/Desktop/PhD/RELIABLE_NAS/NOTEBOOK/FAULT_INJECTOR/VITTFI')

import torch
import numpy as np
from torch.utils.data import DataLoader

from utils import get_network, get_loader
from FaultGenerators.WeightFaultInjector import WeightFaultInjector
from BERCampaign import BERCampaign

In [9]:
ROOT        = '/home/nicolo_b/Desktop/PhD/RELIABLE_NAS/NOTEBOOK/FAULT_INJECTOR/VITTFI'
DATASET_ROOT = f'{ROOT}/data'
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NETWORK     = 'VGG13_BN_SGD_EP_200_LR_04'
DATASET     = 'CIFAR100'

network = get_network(network_name=NETWORK, device=DEVICE, dataset_name=DATASET, root=ROOT)
_, loader = get_loader(network_name=NETWORK, batch_size=128, dataset_name=DATASET, root=DATASET_ROOT)

print(f'Device: {DEVICE}')
print(f'Network loaded: {NETWORK}')
print(f'Loader batches: {len(loader)}')

Loading network VGG13_BN_SGD_EP_200_LR_04
Loading CIFAR100 dataset
Dataset loaded
Batch size:		128 
Number of batches:	79
Device: cuda
Network loaded: VGG13_BN_SGD_EP_200_LR_04
Loader batches: 79


In [14]:
injector = WeightFaultInjector(network)

total_bits = 300_865_536  # dai metadati della cella precedente

p_values         = [1e-7, 1e-6, 1e-5]
injection_levels = [max(1, round(p * total_bits)) for p in p_values]

print('Griglia injection levels:')
for p, k in zip(p_values, injection_levels):
    print(f'  p={p:.0e}  →  k={k} bit')

campaign = BERCampaign(
    network          = network,
    loader           = loader,
    injector         = injector,
    device           = DEVICE,
    injection_levels = injection_levels,
    sampling_mode    = 'constant',
    pilot_trials     = 10,
    max_trials       = 50,
    precision_e      = 0.01,
    confidence_t     = 2.576,
    seed             = 51195,
)

Griglia injection levels:
  p=1e-07  →  k=30 bit
  p=1e-06  →  k=301 bit
  p=1e-05  →  k=3009 bit


In [15]:
results = campaign.run()

[WARNING] injection_level=30: effective half-width 0.1094 > precision_e 0.0100. Consider increasing max_trials.
[BERCampaign] level=30 | trials=50 | mean=0.4301 | std=0.3002 | half_width=0.1094
[BERCampaign] level=301 | trials=10 | mean=0.0115 | std=0.0028 | half_width=0.0023
[BERCampaign] level=3009 | trials=10 | mean=0.0095 | std=0.0017 | half_width=0.0014


In [16]:
golden = list(results.values())[0]['golden_accuracy']
print(f'golden_accuracy = {golden:.4f}\n')

for level, r in results.items():
    print(f"injection_level = {level}")
    print(f"  mean faulty acc      = {r['mean']:.4f}")
    print(f"  std                  = {r['std']:.4f}")
    print(f"  n_trials             = {r['n_trials']}")
    print(f"  effective_half_width = {r['effective_half_width']:.4f}")
    print()

golden_accuracy = 0.7138

injection_level = 30
  mean faulty acc      = 0.4301
  std                  = 0.3002
  n_trials             = 50
  effective_half_width = 0.1094

injection_level = 301
  mean faulty acc      = 0.0115
  std                  = 0.0028
  n_trials             = 10
  effective_half_width = 0.0023

injection_level = 3009
  mean faulty acc      = 0.0095
  std                  = 0.0017
  n_trials             = 10
  effective_half_width = 0.0014

